# AKI: revision analyses for IBMED-D-26-00792

Version 2: every belief score comes from the published JAX engine in float32, so the AUROCs
reproduce the manuscript exactly (version 1 used a float64 re-implementation).

Standalone notebook. It imports the configuration, the JAX engine, the KDIGO label builder,
the Platt calibrator and the patient bootstrap **verbatim** from `AKI_PEMDT_KDIGO.ipynb`
(definitions only; nothing in that notebook is re-run), loads `derived/aki_cohort_v2.parquet`,
and reproduces the published grouped split (seed 7, 30% of subjects held out).

| Section | Reviewer point |
|---|---|
| R1 calibration intercept/slope, raw vs recalibrated, 12/24/48 h | R1.4, R2.3, minor (CIs) |
| R2 operating characteristics, alerts per patient-day, decision curve | R1.5, R2.2 |
| R3 ablation (ordering, weights, age, dynamics, anchoring, RRT) | R1.3, R1.8 |
| R4 baselines (LR, RF, gradient boosting, GRU) on identical split and labels | R1.2 |
| R5 temporal validation by MIMIC-IV anchor_year_group | R1.10 |
| R6 attribution fidelity and stability | R1.7 |
| R7 computational cost and reproducibility record | R1.9, R2.1 |

All outputs go to `results/revision_v2/` (git-ignored; derived from MIMIC-IV, do not commit).
Expected run time on a laptop: about 30 to 60 minutes (most of it R4).

In [1]:
# ---------------------------------------------------------------- R0 · setup
import os, sys, time, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd().parent))          # JBI/revision_utils.py
import revision_utils as ru
os.environ["JAX_PLATFORMS"] = "cpu"
import jax.numpy as jnp

SRC_NB = Path("AKI_PEMDT_KDIGO.ipynb")
REV = Path("results/revision_v2"); REV.mkdir(parents=True, exist_ok=True)
FIGR = REV / "figures"; FIGR.mkdir(exist_ok=True)
SYNTHETIC_TEST = os.environ.get("REV_SYNTHETIC", "0") == "1"   # smoke test only, never for results

# Definitions copied from the original notebook at run time (single source of truth)
exec(ru.cell_source(SRC_NB, "1 · CONFIG"), globals())
exec(ru.cell_source(SRC_NB, "2 · LAYER 1"), globals())
exec(ru.defs_only(ru.cell_source(SRC_NB, "6 · RUN LAYER 1")), globals())
exec(ru.defs_only(ru.cell_source(SRC_NB, "8 · VALIDATION")), globals())

N_BOOT = int(os.environ.get("REV_BOOT", "200"))
SEED = EVAL.random_state                                # 7, as in the paper
HORIZONS = (12, 24, 48)
print("Environment:", ru.env_versions())

Config loaded (v3.3-faithful + label/ETL fixes).
  States       : ['No_AKI', 'KDIGO_1', 'KDIGO_2', 'KDIGO_3'] | emission mode: clinical
  Output        : aki_cohort_v2.parquet (v2 — stale v1 parquet is not loaded)
  MIMIC_ROOT   : /Users/sumiyasayeed/Documents/mimic4/mimiciv (exists: True )
Layer-1 engine ported (JAX forward_filter, default_state_means/scales/transition,
run_filter_patient, compute_global_stats, attribute_window).  Devices: [CpuDevice(id=0)]
Environment: {'python': '3.13.5', 'platform': 'macOS-27.0.1-arm64-arm-64bit-Mach-O', 'numpy': '2.5.4', 'pandas': '2.3.3', 'scipy': '1.18.1', 'sklearn': '1.9.1', 'jax': '0.10.0', 'xgboost': '3.2.0', 'torch': '2.13.0', 'lifelines': '0.30.3'}


In [2]:
# ------------------------------------------- R0b · cohort, UO smoothing, split
NEED = ["stay_id", "subject_id", "anchor_age", "hour", "creatinine", "creatinine_ratio",
        "dSCr_48h", "uo_rate", "bun", "rrt_on", "kdigo_stage", "baseline_method"]
if SYNTHETIC_TEST:
    exec(ru.defs_only(ru.cell_source(SRC_NB, "5 · SYNTHETIC AKI COHORT")), globals())
    C = make_synthetic_cohort(n_stays=int(os.environ.get("REV_NSTAYS", "600")))
    C["baseline_method"] = "synthetic"
else:
    C = pd.read_parquet(COHORT_PARQUET, columns=NEED)
C = smooth_uo_emission(C)                 # same causal 6 h trailing mean as the paper (sorts stay, hour)
C = C.reset_index(drop=True)
C["_row"] = np.arange(len(C))
_tr, _te = split_by_subject(C[["subject_id"]])   # identical subject order => identical split
TEST_SUBJ = set(_te["subject_id"].unique())
C["is_test"] = C["subject_id"].isin(TEST_SUBJ)
assert not (set(_tr["subject_id"]) & TEST_SUBJ), "subject leakage between train and test"
gstats_train = compute_global_stats(C[~C.is_test])
print(f"Stays {C.stay_id.nunique():,} | stay-hours {len(C):,} | test subjects {len(TEST_SUBJ):,} "
      f"| test stays {C.loc[C.is_test,'stay_id'].nunique():,}")

# stay layout on the flat arrays (rows are sorted by stay_id, hour)
_sid = C["stay_id"].to_numpy()
STARTS = np.r_[0, np.nonzero(np.diff(_sid))[0] + 1]
LENS = np.diff(np.r_[STARTS, len(C)])
STAY_AGE_IDX = np.array([age_bracket(a if np.isfinite(a) else 65.0)
                         for a in C["anchor_age"].to_numpy(float)[STARTS]])
ROW_START = np.repeat(STARTS, LENS)                 # first row of the stay, per row
RRT_ON = pd.to_numeric(C["rrt_on"], errors="coerce").fillna(0).to_numpy() > 0

# KDIGO escalation labels from the paper's own function, computed once per horizon
LAB = {}
for H in HORIZONS:
    LAB[H] = build_escalation_labels(C[["_row", "stay_id", "subject_id", "hour", "kdigo_stage",
                                        "is_test"]], horizon_h=H, target_stage=2)
    print(f"H={H:>2} h: below-stage-2 hours {len(LAB[H]):,}, prevalence {LAB[H].y_escalate.mean():.3f} "
          f"(test {LAB[H].loc[LAB[H].is_test,'y_escalate'].mean():.3f})")

Stays 52,742 | stay-hours 6,616,131 | test subjects 12,227 | test stays 15,820
H=12 h: below-stage-2 hours 5,582,871, prevalence 0.039 (test 0.039)
H=24 h: below-stage-2 hours 5,582,871, prevalence 0.065 (test 0.065)
H=48 h: below-stage-2 hours 5,582,871, prevalence 0.094 (test 0.094)


In [3]:
# ---------------- R0c · published JAX engine (float32) for all variants + equivalence check
FEATS = list(FEATURES)
W_FULL = np.array([FEATURE_WEIGHTS[f] for f in FEATS], float)


def obs_matrix(mode="clinical", gstats=None):
    X = np.zeros((len(C), len(FEATS))); M = np.zeros_like(X)
    for j, f in enumerate(FEATS):
        raw = pd.to_numeric(C[f], errors="coerce").to_numpy(float)
        lo, hi = VALID_RANGES.get(f, (-np.inf, np.inf))
        ok = ~np.isnan(raw) & (raw > lo) & (raw < hi)
        if mode == "clinical":
            X[ok, j] = raw[ok]
        else:
            mu, sd = (gstats or GLOBAL_STATS)[f]; X[ok, j] = (raw[ok] - mu) / sd
        M[ok, j] = 1.0
    return X, M


def feature_loglik(X, M, means, scales, w):
    """Per-feature weighted Gaussian log-likelihood, shape (N, K, F); 0 where unobserved."""
    out = np.empty((len(X), means.shape[0], X.shape[1]), dtype=np.float64)
    for a in range(0, len(X), 1_000_000):
        x = X[a:a + 1_000_000, None, :]
        lp = -0.5 * np.log(2 * np.pi) - np.log(scales)[None] - 0.5 * ((x - means[None]) / scales[None]) ** 2
        out[a:a + 1_000_000] = lp * (M[a:a + 1_000_000] * w[None, :])[:, None, :]
    return out


def unordered(A):
    """Keep each self-transition, spread the exit mass uniformly over all other states."""
    K = A.shape[0]; U = np.empty_like(A)
    for k in range(K):
        U[k] = (1 - A[k, k]) / (K - 1); U[k, k] = A[k, k]
    return U


def transition_bank(kind="tridiag"):
    bank = np.stack([default_transition(a) for a in range(len(AGE_BRACKETS))])
    if kind == "no_age":
        bank = np.stack([default_transition(1)] * len(AGE_BRACKETS))
    if kind == "unordered":
        bank = np.stack([unordered(A) for A in bank])
    return bank


def log_emission(X, M, means, scales, w, chunk=500_000):
    """Summed weighted log-emission per state, shape (N, K), computed in row chunks."""
    out = np.empty((len(X), means.shape[0]))
    for a in range(0, len(X), chunk):
        out[a:a + chunk] = feature_loglik(X[a:a + chunk], M[a:a + chunk], means, scales, w).sum(2)
    return out


def _bucket(n):
    """Pad each stay to the next power of two so JAX compiles ~15 shapes instead of one per
    stay length. The filter is causal, so trailing padded rows (mask 0) cannot change the
    beliefs of the real rows; this is verified below against the unpadded published call."""
    return int(2 ** np.ceil(np.log2(max(int(n), 8))))


def filter_jax(X, M, means, scales, bank, weights, static=False, prior=PRIOR_BELIEF):
    """The published JAX `forward_filter` (float32, exactly as in the paper), run per stay."""
    out = np.empty((len(X), N_STATES), np.float32)
    jm, js = jnp.array(means), jnp.array(scales)
    jw, jp = jnp.array(weights), jnp.array(prior)
    # static: every transition row equals the prior, so the prediction step returns the prior
    jb = [jnp.array(np.tile(prior, (N_STATES, 1)) if static else A) for A in bank]
    F = X.shape[1]
    for k in range(len(STARTS)):
        st, ln = int(STARTS[k]), int(LENS[k]); Lb = _bucket(ln)
        x = np.zeros((Lb, F)); m = np.zeros((Lb, F))
        x[:ln] = X[st:st + ln]; m[:ln] = M[st:st + ln]
        b = forward_filter(jnp.array(x), jnp.array(m), jm, js, jb[STAY_AGE_IDX[k]], jp, jw)
        out[st:st + ln] = np.asarray(b)[:ln]
    return out


def risk_of(B):
    """p(KDIGO >= 2) exactly as run_filter_patient computes it (float32 sum, clipped)."""
    return np.clip(B[:, 2] + B[:, 3], 0.0, 1.0)


def run_variant(name):
    cfg = dict(mode="clinical", w=W_FULL, bank="tridiag", static=False, rrt=True)
    cfg.update(VARIANTS[name])
    X, M = obs_matrix(cfg["mode"], gstats_train)
    means, scales = (CLINICAL_MEANS, CLINICAL_SCALES) if cfg["mode"] == "clinical" \
        else (STATE_MEANS_POP, STATE_SCALES_POP)
    B = np.clip(filter_jax(X, M, means, scales, transition_bank(cfg["bank"]), cfg["w"],
                           static=cfg["static"]), 0.0, 1.0)
    Braw = B.copy()
    if cfg["rrt"]:
        B[RRT_ON] = np.eye(N_STATES, dtype=np.float32)[ABSORBING_RRT_STATE]
    return B, Braw


VARIANTS = {
    "Full model":                    {},
    "Equal feature weights":         dict(w=np.ones(len(FEATS))),
    "Unordered transitions":         dict(bank="unordered"),
    "No age-specific recovery":      dict(bank="no_age"),
    "No temporal dynamics (static)": dict(static=True),
    "Population z-score emissions":  dict(mode="zscore"),
    "No RRT absorbing state":        dict(rrt=False),
}

t0 = time.time()
B_FULL, B_FULL_RAW = run_variant("Full model")
T_ENGINE = time.time() - t0
C["risk_eng"] = risk_of(B_FULL)
print(f"published JAX engine (float32): {len(C):,} stay-hours in {T_ENGINE:.0f} s "
      f"({1e6 * T_ENGINE / len(C):.2f} µs per stay-hour, including JIT compilation)")

# the padded batch run must reproduce the published per-stay call exactly
_rng = np.random.default_rng(SEED)
_chk = _rng.choice(C["stay_id"].unique(), size=min(200, C.stay_id.nunique()), replace=False)
diffs = []
_groups = [C[C.stay_id == sid].sort_values("hour") for sid in _chk]
for g in _groups:
    r = run_filter_patient(g, global_stats=gstats_train)
    diffs.append(np.max(np.abs(r["risk_score"].to_numpy() - g["risk_eng"].to_numpy())))
t0 = time.time()
for g in _groups:
    run_filter_patient(g, global_stats=gstats_train)
T_JAX_PER_STAY = (time.time() - t0) / len(_groups)
MAXDIFF = float(np.max(diffs))
print(f"max |batch run - published run_filter_patient| over {len(_chk)} stays = {MAXDIFF:.2e}")
assert MAXDIFF < 1e-6, "batch run does not reproduce the published engine; stop and send this output"

published JAX engine (float32): 6,616,131 stay-hours in 5 s (0.78 µs per stay-hour, including JIT compilation)
max |batch run - published run_filter_patient| over 200 stays = 0.00e+00


In [4]:
# ------------- R1-audit · resolution of the raw belief score (decides the recalibration)
# Paste after R0c. Uses C, LAB, HORIZONS, REV, ru. Refits only the paper's Platt map on
# TRAINING hours, never looks at test-patient outcomes, writes two small aggregate CSVs.
def _logit(v):
    v = np.asarray(v, float)
    return np.log(v) - np.log1p(-v)


def _group_codes(s):
    """0 = exact 0, 1..10 = deciles of log-odds among 0 < s < 1, 11 = exact 1."""
    code = np.zeros(len(s), int)
    mid = (s > 0) & (s < 1)
    if mid.any():
        code[mid] = pd.qcut(_logit(s[mid]), 10, labels=False, duplicates="drop") + 1
    code[s >= 1] = 11
    return code


_lab = {0: "exact 0", 11: "exact 1"}
aud, dec = [], []
for H in HORIZONS:
    L = LAB[H]
    s32 = C["risk_eng"].to_numpy()[L["_row"].to_numpy()]            # published float32 score
    s = s32.astype(float)
    vals, cnt = np.unique(s, return_counts=True)
    aud.append(dict(horizon_h=H, dtype=str(s32.dtype), hours=len(s),
                    exact_zero=np.mean(s == 0), below_1e_12=np.mean(s < 1e-12),
                    below_1e_8=np.mean(s < 1e-8), below_1e_6=np.mean(s < 1e-6),
                    above_1_minus_1e_7=np.mean(s > 1 - 1e-7), exact_one=np.mean(s == 1),
                    distinct_values=len(vals), largest_tie_share=cnt.max() / len(s),
                    value_of_largest_tie=float(vals[cnt.argmax()])))
    tr = ~L["is_test"].to_numpy()
    st, yt = s[tr], L["y_escalate"].to_numpy()[tr]
    p_now = ru.platt_apply(ru.platt_fit(st, yt), st)                 # the map used in the paper
    t = (pd.DataFrame({"g": _group_codes(st), "s": st, "y": yt, "p": p_now})
         .groupby("g").agg(hours=("y", "size"), raw_median=("s", "median"),
                           current_prob=("p", "mean"), observed=("y", "mean")))
    t["share"] = t["hours"] / len(st)
    t.index = [_lab.get(k, f"log-odds decile {k}") for k in t.index]
    dec.append(t.reset_index(names="group").assign(horizon_h=H))

AUD = pd.DataFrame(aud)
DEC = pd.concat(dec, ignore_index=True)[["horizon_h", "group", "hours", "share", "raw_median",
                                         "current_prob", "observed"]]
AUD.to_csv(REV / "R1_audit_score_resolution.csv", index=False)
DEC.to_csv(REV / "R1_audit_training_groups.csv", index=False)
pd.set_option("display.width", 200)
print("Raw-score resolution, all below-stage-2 hours (no outcomes used):")
print(AUD.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("\nTraining hours only: observed rate by group vs the current Platt probability:")
print(DEC.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

Raw-score resolution, all below-stage-2 hours (no outcomes used):
 horizon_h   dtype   hours  exact_zero  below_1e_12  below_1e_8  below_1e_6  above_1_minus_1e_7  exact_one  distinct_values  largest_tie_share  value_of_largest_tie
        12 float32 5582871      0.5299       0.7518      0.8571      0.8997            0.001918   0.001453          2057230             0.5299                     0
        24 float32 5582871      0.5299       0.7518      0.8571      0.8997            0.001918   0.001453          2057230             0.5299                     0
        48 float32 5582871      0.5299       0.7518      0.8571      0.8997            0.001918   0.001453          2057230             0.5299                     0

Training hours only: observed rate by group vs the current Platt probability:
 horizon_h              group   hours    share  raw_median  current_prob  observed
        12            exact 0 2068517   0.5315           0       0.03648  0.008778
        12  log-odds decile 1

In [5]:
# ======================================================================================
# R8 · Recalibration on the log-odds scale: pre-specified comparison (kidney injury)
# --------------------------------------------------------------------------------------
# Paste as a new cell after R0c (R1-R7 are not needed). Uses C, LAB, HORIZONS, ru, REV,
# FIGR, N_BOOT, SEED. Writes only new files (R8_*); nothing existing is overwritten.
#
# Pre-specified plan
#   current map : logit p = a + b * s                 (Platt on the raw belief score s)
#   log-odds map: logit p = a + b * logit(s*)         s* = s clipped to [2^-149, 1 - 2^-24],
#                 the float32 resolution limits, so the 53% of hours with s == 0 form one
#                 lowest-risk group and the map stays strictly increasing (AUROC unchanged).
#   Step 1  choose on TRAINING patients only: 5-fold grouped (by subject) cross-validation,
#           out-of-fold ICI and Brier. Rule: keep the log-odds map if its out-of-fold ICI is
#           lower at every horizon; otherwise keep the current map.
#   Step 2  fit the chosen map on all training hours, evaluate ONCE on test patients.
#   Step 3  same comparison under temporal validation (development years -> latest years).
#   ICI (Austin and Steyerberg, Stat Med 2019): mean |p - E[y | p]|. E[y | p] is estimated by
#   logistic regression of y on a restricted cubic spline (4 knots at the 5th, 35th, 65th and
#   95th percentiles of logit(s*) among s > 0) plus an indicator for s == 0, fitted on the
#   evaluation data. Both maps are strictly increasing functions of s, so E[y | p] = E[y | s]
#   and ONE smoother serves both maps. A spline replaces loess because loess is not feasible
#   on 1.7 million hours. CIs: patient-level bootstrap, smoother refitted in every replicate.
# ======================================================================================
from sklearn.model_selection import GroupKFold

_LO, _HI = 2.0 ** -149, 1.0 - 2.0 ** -24          # float32 smallest subnormal; largest value < 1


def _lgt(s):
    s = np.clip(np.asarray(s, float), _LO, _HI)
    return np.log(s) - np.log1p(-s)


def _rcs(x, knots):
    """Restricted cubic spline basis (Harrell), k knots -> k-2 nonlinear columns."""
    t = np.asarray(knots, float); k = len(t); sc = (t[-1] - t[0]) ** 2
    p3 = lambda u: np.maximum(u, 0.0) ** 3
    cols = [(p3(x - t[j]) - p3(x - t[k - 2]) * (t[k - 1] - t[j]) / (t[k - 1] - t[k - 2])
             + p3(x - t[k - 1]) * (t[k - 2] - t[j]) / (t[k - 1] - t[k - 2])) / sc for j in range(k - 2)]
    return np.column_stack(cols)


class _Smoother:
    """E[y | s] for ICI: logistic on [1, zero, x, rcs(x)] with x = logit(s*) for s > 0."""
    def __init__(self, s):
        s = np.asarray(s, float); self.zero = (s == 0).astype(float); nz = s > 0
        x = _lgt(s)
        self.knots = np.unique(np.percentile(x[nz], [5, 35, 65, 95]))
        nzf = 1.0 - self.zero
        cols = [np.ones_like(x), self.zero, x * nzf]
        if len(self.knots) >= 3:
            cols.append(_rcs(x, self.knots) * nzf[:, None])
        self.X = np.column_stack(cols)

    def fit_predict(self, y, w=None):
        b = ru._irls(self.X, np.asarray(y, float), w)
        return 1.0 / (1.0 + np.exp(-np.clip(self.X @ b, -30, 30)))


def _ici(p, m, w=None):
    w = np.ones(len(p)) if w is None else w
    return float(np.sum(w * np.abs(p - m)) / np.sum(w))


def _maps(s_fit, y_fit):
    """Fit both maps on (s_fit, y_fit); return a dict name -> function(s) -> probability."""
    b_cur = ru.platt_fit(s_fit, y_fit)
    b_log = ru.platt_fit(_lgt(s_fit), y_fit)
    assert b_log[1] > 0, "log-odds map is not increasing; stop and send this output"
    return {"current (Platt on s)": lambda s: ru.platt_apply(b_cur, s),
            "log-odds (Platt on logit s)": lambda s: ru.platt_apply(b_log, _lgt(s))}, b_cur, b_log


def _ici_boot(y, s, preds, subject, n_boot=N_BOOT, seed=SEED):
    """ICI point estimate and patient-bootstrap 95% CI for every map, one shared smoother."""
    sm = _Smoother(s)
    m = sm.fit_predict(y)
    point = {k: _ici(p, m) for k, p in preds.items()}
    boots = {k: [] for k in preds}
    for w in ru.PatientBootstrap(subject, n_boot, seed).weights():
        if np.sum(w * y) == 0 or np.sum(w * (1 - y)) == 0:
            continue
        mb = sm.fit_predict(y, w)
        for k, p in preds.items():
            boots[k].append(_ici(p, mb, w))
    return {k: (point[k], *np.percentile(boots[k], [2.5, 97.5])) for k in preds}


def _groups(s):
    """Same grouping as the audit: exact 0, ten log-odds deciles among 0 < s < 1, exact 1."""
    code = np.zeros(len(s), int); mid = (s > 0) & (s < 1)
    if mid.any():
        code[mid] = pd.qcut(_lgt(s[mid]), 10, labels=False, duplicates="drop") + 1
    code[s >= 1] = 11
    return code


# episode metrics, identical to R2 (copied so that R2 need not be run in this session)
_tst = C[C.is_test]
_sev = _tst[_tst.kdigo_stage >= 2].groupby("stay_id")["hour"].min()
_first_below = _tst[_tst.kdigo_stage < 2].groupby("stay_id")["hour"].min()
_FIRST_ONSET = _sev[_sev > _first_below.reindex(_sev.index).fillna(np.inf)]


def _episode_metrics(L, p, thr, H):
    d = L.assign(_p=p).sort_values(["stay_id", "hour"])
    y = d.y_escalate.to_numpy(); pp = d["_p"].to_numpy()
    st = d.stay_id.to_numpy(); hr = d.hour.to_numpy()
    a = pp >= thr
    new_seg = np.r_[True, (st[1:] != st[:-1]) | (np.diff(hr) != 1)]
    onset = a & ~(np.r_[False, a[:-1]] & ~new_seg)
    pdays = len(d) / 24.0
    res = ru.confusion_at(y, pp, thr)
    res.update(false_alert_episodes_per_patient_day=(onset & (y == 0)).sum() / pdays)
    ev = _FIRST_ONSET.loc[_FIRST_ONSET.index.isin(np.unique(st))]
    dd = d.assign(alert=a).merge(ev.rename("t_on"), left_on="stay_id", right_index=True)
    win = dd[(dd.hour >= dd.t_on - H) & (dd.hour < dd.t_on)]
    cap = win.groupby("stay_id")["alert"].any()
    first = win[win.alert].groupby("stay_id")["hour"].min()
    lead = ev.loc[first.index] - first
    res.update(event_sensitivity=float(cap.mean()) if len(cap) else np.nan,
               median_lead_h=float(lead.median()) if len(lead) else np.nan)
    return res


# ----------------------------------------------------------- step 1: training-only CV
cv_rows, test_rows, ici_rows, op_rows, grp_rows, R8_MAP = [], [], [], [], [], {}
t_all = time.time()
for H in HORIZONS:
    L = LAB[H]
    s = C["risk_eng"].to_numpy()[L["_row"].to_numpy()].astype(float)
    y = L["y_escalate"].to_numpy().astype(int)
    subj = L["subject_id"].to_numpy()
    tr, te = ~L["is_test"].to_numpy(), L["is_test"].to_numpy()
    s_tr, y_tr, g_tr = s[tr], y[tr], subj[tr]
    oof = {k: np.empty(tr.sum()) for k in ["current (Platt on s)", "log-odds (Platt on logit s)"]}
    fold_id = np.empty(tr.sum(), int)
    for f, (a, b) in enumerate(GroupKFold(n_splits=5).split(s_tr, y_tr, groups=g_tr)):
        maps, _, _ = _maps(s_tr[a], y_tr[a])
        for k, fn in maps.items():
            oof[k][b] = fn(s_tr[b])
        fold_id[b] = f
    m_oof = _Smoother(s_tr).fit_predict(y_tr)
    for k, p in oof.items():
        per_fold = [_ici(p[fold_id == f], _Smoother(s_tr[fold_id == f]).fit_predict(y_tr[fold_id == f]))
                    for f in range(5)]
        c = ru.calibration_stats(y_tr, p)
        cv_rows.append(dict(horizon_h=H, map=k, oof_ICI=_ici(p, m_oof), fold_ICI_mean=np.mean(per_fold),
                            fold_ICI_sd=np.std(per_fold), oof_brier=c["brier"], oof_intercept=c["citl"],
                            oof_slope=c["slope"], oof_EO=c["eo"]))
    print(f"H={H:>2} h: training cross-validation done ({time.time() - t_all:.0f} s)")

CV = pd.DataFrame(cv_rows)
_piv = CV.pivot(index="horizon_h", columns="map", values="oof_ICI")
CHOOSE_LOGODDS = bool((_piv["log-odds (Platt on logit s)"] < _piv["current (Platt on s)"]).all())
print("\nPre-specified rule (training patients only):",
      "log-odds map chosen" if CHOOSE_LOGODDS else "current map kept")

# --------------------------------------- step 2: fit on all training hours, test once
for H in HORIZONS:
    L = LAB[H]
    s = C["risk_eng"].to_numpy()[L["_row"].to_numpy()].astype(float)
    y = L["y_escalate"].to_numpy().astype(int)
    subj = L["subject_id"].to_numpy()
    tr, te = ~L["is_test"].to_numpy(), L["is_test"].to_numpy()
    maps, b_cur, b_log = _maps(s[tr], y[tr])
    R8_MAP[H] = dict(current=b_cur, logodds=b_log)
    P = {k: fn(s) for k, fn in maps.items()}
    for k, p in P.items():
        e = ru.evaluate(y[te], p[te], subj[te], n_boot=N_BOOT, seed=SEED, label=f"{H} h | {k}",
                        rank_score=s[te])
        test_rows.append(e)
    # the map must not change discrimination: AUROC on the probability itself equals raw AUROC
    _ss_raw, _ss_new = ru.SortedScores(s[te]), ru.SortedScores(P["log-odds (Platt on logit s)"][te])
    _one = np.ones(te.sum())
    auc_raw = ru.w_auc_ap(y[te], _one, _ss_raw)[0]; auc_new = ru.w_auc_ap(y[te], _one, _ss_new)[0]
    assert abs(auc_raw - auc_new) < 1e-6, "log-odds map changed the ranking; stop and send this output"
    for k, (est, lo, hi) in _ici_boot(y[te], s[te], {k: p[te] for k, p in P.items()}, subj[te]).items():
        ici_rows.append(dict(horizon_h=H, map=k, ICI=est, lo=lo, hi=hi))
    # operating points: train-derived 90% specificity (must be unchanged) and fixed thresholds
    Lte = L[te]
    for k, p in P.items():
        thr90 = ru.threshold_for_specificity(y[tr], p[tr], 0.90)
        for rule, t in [("90% specificity (train)", thr90), ("fixed 0.05", 0.05), ("fixed 0.10", 0.10),
                        ("fixed 0.20", 0.20), ("fixed 0.30", 0.30)]:
            r = _episode_metrics(Lte, p[te], t, H); r.update(horizon_h=H, map=k, rule=rule); op_rows.append(r)
    # reliability groups on test hours (same grouping for both maps)
    g = _groups(s[te])
    for k, p in [("raw belief score", s)] + list(P.items()):
        t = pd.DataFrame({"g": g, "p": p[te], "y": y[te]}).groupby("g").agg(
            hours=("y", "size"), predicted=("p", "mean"), observed=("y", "mean")).reset_index()
        grp_rows.append(t.assign(horizon_h=H, map=k))
    print(f"H={H:>2} h: test evaluation done ({time.time() - t_all:.0f} s)")

TEST = pd.concat(test_rows, ignore_index=True)
ICI = pd.DataFrame(ici_rows)
OPS = pd.DataFrame(op_rows)[["horizon_h", "map", "rule", "threshold", "sensitivity", "specificity", "PPV",
                             "NPV", "false_alert_episodes_per_patient_day", "event_sensitivity", "median_lead_h"]]
GRP = pd.concat(grp_rows, ignore_index=True)

# ------------------------------------------------ step 3: temporal validation (24 h)
TEMP = None
try:
    if "AYG" not in globals():
        _cands = [os.environ.get("MIMIC_HOSP", ""), str(globals().get("HOSP_DIR", "")),
                  "/Users/sumiyasayeed/Documents/mimic4/mimiciv/hosp", "/Users/sumiyasayeed/Documents/mimic4/hosp"]
        _pc = next((Path(d) / f"patients{e}" for d in _cands for e in (".csv", ".csv.gz")
                    if d and (Path(d) / f"patients{e}").exists()), None)
        if _pc is None:
            _hits = sorted(Path("/Users/sumiyasayeed/Documents/mimic4").glob("**/patients.csv*"))
            _pc = _hits[0] if _hits else None
        if _pc is None:
            raise FileNotFoundError("patients.csv(.gz) not found; set os.environ['MIMIC_HOSP'] and rerun")
        AYG = ru.anchor_year_groups(_pc).reindex(C.subject_id.unique())
    DEV_G, TEST_G = ru.temporal_split(AYG, 0.10)
    H = 24; L = LAB[H]
    ayg = L["subject_id"].map(AYG).to_numpy()
    dv, tt = np.isin(ayg, DEV_G), np.isin(ayg, TEST_G)
    s = C["risk_eng"].to_numpy()[L["_row"].to_numpy()].astype(float)
    y = L["y_escalate"].to_numpy().astype(int); subj = L["subject_id"].to_numpy()
    maps, _, _ = _maps(s[dv], y[dv])
    P = {k: fn(s[tt]) for k, fn in maps.items()}
    rows = [ru.evaluate(y[tt], p, subj[tt], n_boot=N_BOOT, seed=SEED, label=f"temporal 24 h | {k}",
                        rank_score=s[tt]) for k, p in P.items()]
    TEMP = pd.concat(rows, ignore_index=True)
    for k, (est, lo, hi) in _ici_boot(y[tt], s[tt], P, subj[tt]).items():
        ici_rows.append(dict(horizon_h="temporal 24", map=k, ICI=est, lo=lo, hi=hi))
    ICI = pd.DataFrame(ici_rows)
    print(f"temporal validation done: development {DEV_G}, test {TEST_G}")
except Exception as exc:                                     # report, do not stop the cell
    print("temporal validation skipped:", exc)

# -------------------------------------------------------------------- figures
_MC = {"current (Platt on s)": dict(color="0.55", marker="o", mfc="white"),
       "log-odds (Platt on logit s)": dict(color="#1f77b4", marker="o")}
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for i, H in enumerate(HORIZONS):
    for k, st_ in _MC.items():
        t = GRP[(GRP.horizon_h == H) & (GRP["map"] == k)]
        ax[i].plot(t.predicted, t.observed, "-", lw=1.2, label=k.split(" (")[0] + " map", **st_)
    m = GRP[(GRP.horizon_h == H) & GRP["map"].isin(list(_MC))][["predicted", "observed"]].max().max() * 1.08
    z = GRP[(GRP.horizon_h == H) & (GRP["map"] == "log-odds (Platt on logit s)") & (GRP.g == 0)]
    if len(z):
        share = z.hours.iloc[0] / GRP[(GRP.horizon_h == H) & (GRP["map"] == "log-odds (Platt on logit s)")].hours.sum()
        ax[i].annotate(f"raw score 0\n({share:.0%} of hours)", (z.predicted.iloc[0], z.observed.iloc[0]),
                       xytext=(12, 18), textcoords="offset points", fontsize=8,
                       arrowprops=dict(arrowstyle="-", color="0.4", lw=0.8))
    ax[i].plot([0, m], [0, m], "--", color="grey", lw=1)
    ax[i].set(xlim=(0, m), ylim=(0, m), title=f"Stage ≥2 within {H} h", xlabel="Predicted", ylabel="Observed")
ax[0].legend(loc="upper left", frameon=False)
plt.tight_layout(); plt.savefig(FIGR / "R8_aki_reliability_current_vs_logodds.png", dpi=200); plt.close()

H = 24
fig, a = plt.subplots(figsize=(6, 6))
_b = TEST[(TEST.metric == "Brier")].set_index("model")["estimate"]
_raw_brier = ru.calibration_stats(LAB[H].y_escalate.to_numpy()[LAB[H].is_test.to_numpy()],
                                  C["risk_eng"].to_numpy()[LAB[H]["_row"].to_numpy()][LAB[H].is_test.to_numpy()])["brier"]
for k, lab, st_ in [("raw belief score", f"raw belief score (Brier {_raw_brier:.3f})", dict(color="#2ca02c", marker="o")),
                    ("log-odds (Platt on logit s)",
                     f"recalibrated (Brier {_b[f'{H} h | log-odds (Platt on logit s)']:.3f})", dict(color="#1f77b4", marker="o"))]:
    t = GRP[(GRP.horizon_h == H) & (GRP["map"] == k)]
    a.plot(t.predicted, t.observed, "-", lw=2, label=lab, **st_)
a.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="perfect")
a.set(xlim=(0, 1), ylim=(0, 1), xlabel="predicted probability", ylabel="observed frequency",
      title="Acute kidney injury (24 h)")
a.legend(loc="upper left")
plt.tight_layout(); plt.savefig(FIGR / "R8_aki_calibration_24h_raw_vs_recalibrated.png", dpi=150); plt.close()

# -------------------------------------------------------------------- report
CV.to_csv(REV / "R8_cv_training.csv", index=False)
TEST.to_csv(REV / "R8_test_metrics.csv", index=False)
ICI.to_csv(REV / "R8_ici.csv", index=False)
OPS.to_csv(REV / "R8_operating_points.csv", index=False)
GRP.to_csv(REV / "R8_reliability_groups.csv", index=False)
if TEMP is not None:
    TEMP.to_csv(REV / "R8_temporal.csv", index=False)
json.dump({str(h): {k: list(map(float, v)) for k, v in d.items()} for h, d in R8_MAP.items()},
          open(REV / "R8_map_coefficients.json", "w"), indent=2)

_f = lambda x: f"{x:.4g}"
_aud = REV / "R1_audit_score_resolution.csv"
parts = ["# R8 recalibration comparison (kidney injury)\n",
         f"Pre-specified rule on training patients: **{'log-odds map chosen' if CHOOSE_LOGODDS else 'current map kept'}**\n",
         "## Raw-score resolution (from the R1 audit)\n```\n" +
         (pd.read_csv(_aud).to_string(index=False, float_format=_f) if _aud.exists() else "audit file not found") + "\n```\n",
         "## Step 1: training cross-validation (5 grouped folds)\n```\n" + CV.to_string(index=False, float_format=_f) + "\n```\n",
         "## Step 2: held-out test patients\n```\n" + ru.wide(TEST).to_string() + "\n```\n",
         "## ICI with patient-bootstrap 95% CI\n```\n" + ICI.to_string(index=False, float_format=_f) + "\n```\n",
         "## Operating points on test patients\n```\n" + OPS.to_string(index=False, float_format=_f) + "\n```\n"]
if TEMP is not None:
    parts.append("## Step 3: temporal validation (24 h)\n```\n" + ru.wide(TEMP).to_string() + "\n```\n")
parts.append("## Reliability groups (test hours)\n```\n" + GRP.to_string(index=False, float_format=_f) + "\n```\n")
(REV / "R8_recalibration_report.md").write_text("\n".join(parts))

pd.set_option("display.width", 220)
print("\n=== Step 1 · training cross-validation ===");  print(CV.to_string(index=False, float_format=_f))
print("\n=== Step 2 · test patients ===");               print(ru.wide(TEST).to_string())
print("\n=== ICI (95% CI) ===");                          print(ICI.to_string(index=False, float_format=_f))
print("\n=== Operating points (test) ===");               print(OPS.to_string(index=False, float_format=_f))
if TEMP is not None:
    print("\n=== Step 3 · temporal validation ===");     print(ru.wide(TEMP).to_string())
print(f"\nReport: {REV / 'R8_recalibration_report.md'} | figures: R8_*.png in {FIGR} | {time.time() - t_all:.0f} s")


H=12 h: training cross-validation done (11 s)
H=24 h: training cross-validation done (21 s)
H=48 h: training cross-validation done (30 s)

Pre-specified rule (training patients only): log-odds map chosen
H=12 h: test evaluation done (291 s)
H=24 h: test evaluation done (535 s)
H=48 h: test evaluation done (749 s)
temporal validation done: development ['2008 - 2010', '2011 - 2013', '2014 - 2016', '2017 - 2019'], test ['2020 - 2022']

=== Step 1 · training cross-validation ===
 horizon_h                         map  oof_ICI  fold_ICI_mean  fold_ICI_sd  oof_brier  oof_intercept  oof_slope  oof_EO
        12        current (Platt on s)  0.03496          0.035    0.0005459    0.03724     -8.844e-05     0.9983       1
        12 log-odds (Platt on logit s)  0.00881       0.008841    0.0008084    0.03529      1.973e-05     0.9998       1
        24        current (Platt on s)  0.04885        0.04891    0.0006109    0.06011     -5.755e-05     0.9976       1
        24 log-odds (Platt on logit 

In [6]:
# ------------------ R1 · discrimination and calibration (raw vs recalibrated), 12/24/48 h
rows, PLATT = [], {}
for H in HORIZONS:
    L = LAB[H]
    s = C["risk_eng"].to_numpy()[L["_row"].to_numpy()]
    tr, te = ~L.is_test.to_numpy(), L.is_test.to_numpy()
    PLATT[H] = ru.platt_fit(s[tr], L.y_escalate.to_numpy()[tr])
    p_cal = ru.platt_apply(PLATT[H], s)
    LAB[H]["risk"], LAB[H]["p_cal"] = s, p_cal
    for nm, sc in [("raw belief score", s), ("recalibrated (Platt, train)", p_cal)]:
        e = ru.evaluate(L.y_escalate.to_numpy()[te], sc[te], L.subject_id.to_numpy()[te],
                        n_boot=N_BOOT, seed=SEED, label=f"{H} h | {nm}", rank_score=s[te])
        rows.append(e)
R1 = pd.concat(rows, ignore_index=True)
R1.to_csv(REV / "R1_calibration_by_horizon.csv", index=False)
print(ru.wide(R1).to_string())

# reproduction check against the published manuscript (Results, AKI)
PUBLISHED_AUROC = {12: 0.810, 24: 0.772, 48: 0.738}
for H, pub in PUBLISHED_AUROC.items():
    got = R1.loc[(R1.model == f"{H} h | raw belief score") & (R1.metric == "AUROC"), "estimate"].item()
    print(f"  {H:>2} h AUROC: revision {got:.3f} | published {pub:.3f}")
    if not SYNTHETIC_TEST:
        assert abs(got - pub) < 0.0006, f"AUROC at {H} h does not match the paper; stop and send this output"
print("Published AKI AUROCs reproduced." if not SYNTHETIC_TEST else "(synthetic test: published check skipped)")

# reliability plot, 24 h, deciles of predicted risk
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for i, H in enumerate(HORIZONS):
    L = LAB[H][LAB[H].is_test]
    q = pd.qcut(L["p_cal"], 10, duplicates="drop")
    tab = L.groupby(q, observed=True).agg(pred=("p_cal", "mean"), obs=("y_escalate", "mean"))
    m = max(tab.max().max() * 1.1, 0.05)
    ax[i].plot([0, m], [0, m], "--", color="grey", lw=1)
    ax[i].plot(tab.pred, tab.obs, "o-")
    ax[i].set(title=f"Stage ≥2 within {H} h", xlabel="Predicted", ylabel="Observed")
plt.tight_layout(); plt.savefig(FIGR / "aki_reliability_by_horizon.png", dpi=200); plt.close()

                                                     AUROC                   AUPRC                   Brier                       CITL                   Slope                     E/O
model                                                                                                                                                                                
12 h | raw belief score             0.810 (0.806 to 0.816)  0.167 (0.159 to 0.174)  0.043 (0.041 to 0.045)     6.947 (6.747 to 7.141)  0.150 (0.142 to 0.158)  0.289 (0.261 to 0.315)
12 h | recalibrated (Platt, train)  0.810 (0.806 to 0.816)  0.167 (0.159 to 0.174)  0.037 (0.036 to 0.038)    0.002 (-0.039 to 0.034)  1.030 (0.983 to 1.076)  0.999 (0.968 to 1.038)
24 h | raw belief score             0.772 (0.768 to 0.778)  0.221 (0.211 to 0.230)  0.067 (0.064 to 0.069)     9.033 (8.896 to 9.161)  0.155 (0.146 to 0.165)  0.174 (0.158 to 0.190)
24 h | recalibrated (Platt, train)  0.772 (0.768 to 0.778)  0.221 (0.211 to 0.230)  0.060 

In [7]:
# ---------------------- R2 · operating characteristics, alert burden, decision curve
def episode_and_event_metrics(L, pcol, thr, H):
    """Hour-level confusion matrix plus alert burden per patient-day and event-level capture.
    Alert episode = run of consecutive alerting below-stage-2 hours within a stay.
    False episode = episode whose first hour is not followed by stage >= 2 within H hours."""
    d = L.sort_values(["stay_id", "hour"])
    y = d.y_escalate.to_numpy(); p = d[pcol].to_numpy()
    st = d.stay_id.to_numpy(); hr = d.hour.to_numpy()
    a = p >= thr
    new_seg = np.r_[True, (st[1:] != st[:-1]) | (np.diff(hr) != 1)]
    onset = a & ~(np.r_[False, a[:-1]] & ~new_seg)
    pdays = len(d) / 24.0
    res = ru.confusion_at(y, p, thr)
    res.update(alert_hours_per_patient_day=a.sum() / pdays,
               false_alert_hours_per_patient_day=(a & (y == 0)).sum() / pdays,
               alert_episodes_per_patient_day=onset.sum() / pdays,
               false_alert_episodes_per_patient_day=(onset & (y == 0)).sum() / pdays)
    # event level: first escalation per stay that had below-stage-2 hours before it
    ev = FIRST_ONSET.loc[FIRST_ONSET.index.isin(np.unique(st))]
    dd = d.assign(alert=a).merge(ev.rename("t_on"), left_on="stay_id", right_index=True)
    win = dd[(dd.hour >= dd.t_on - H) & (dd.hour < dd.t_on)]
    cap = win.groupby("stay_id")["alert"].any()
    first = win[win.alert].groupby("stay_id")["hour"].min()
    lead = (ev.loc[first.index] - first)
    res.update(events=int(len(cap)), event_sensitivity=float(cap.mean()) if len(cap) else np.nan,
               median_lead_h=float(lead.median()) if len(lead) else np.nan)
    return res


_tst = C[C.is_test]
_sev = _tst[_tst.kdigo_stage >= 2].groupby("stay_id")["hour"].min()
_first_below = _tst[_tst.kdigo_stage < 2].groupby("stay_id")["hour"].min()
FIRST_ONSET = _sev[_sev > _first_below.reindex(_sev.index).fillna(np.inf)]

rows = []
for H in HORIZONS:
    L = LAB[H]; tr = L[~L.is_test]; te = L[L.is_test]
    thr = [("fixed 0.05", 0.05), ("fixed 0.10", 0.10), ("fixed 0.20", 0.20), ("fixed 0.30", 0.30),
           ("90% specificity (train)", ru.threshold_for_specificity(tr.y_escalate, tr.p_cal, 0.90)),
           ("80% sensitivity (train)", ru.threshold_for_sensitivity(tr.y_escalate, tr.p_cal, 0.80))]
    for nm, t in thr:
        r = episode_and_event_metrics(te, "p_cal", t, H); r.update(horizon_h=H, rule=nm); rows.append(r)
R2 = pd.DataFrame(rows)
_c = ["horizon_h", "rule", "threshold", "sensitivity", "specificity", "PPV", "NPV", "flagged_frac",
      "alert_episodes_per_patient_day", "false_alert_episodes_per_patient_day",
      "false_alert_hours_per_patient_day", "events", "event_sensitivity", "median_lead_h"]
R2 = R2[_c]; R2.to_csv(REV / "R2_operating_characteristics.csv", index=False)
print(R2.round(3).to_string(index=False))

 horizon_h                    rule  threshold  sensitivity  specificity   PPV   NPV  flagged_frac  alert_episodes_per_patient_day  false_alert_episodes_per_patient_day  false_alert_hours_per_patient_day  events  event_sensitivity  median_lead_h
        12              fixed 0.05      0.050        0.090        0.990 0.266 0.964         0.013                           0.051                                 0.038                              0.235    4510              0.174            6.0
        12              fixed 0.10      0.100        0.080        0.992 0.278 0.963         0.011                           0.041                                 0.029                              0.195    4510              0.153            6.0
        12              fixed 0.20      0.200        0.071        0.993 0.286 0.963         0.010                           0.034                                 0.024                              0.167    4510              0.137            6.0
        12          

In [8]:
# ------------------------------------------------------------ R3 · ablation (24 h, test)
H = 24
L = LAB[H]; tr, te = ~L.is_test.to_numpy(), L.is_test.to_numpy()
y = L.y_escalate.to_numpy(); subj = L.subject_id.to_numpy(); rr = L._row.to_numpy()
rows, ABL_SCORES = [], {}
for name in VARIANTS:
    t0 = time.time()
    B = B_FULL if name == "Full model" else run_variant(name)[0]
    s = risk_of(B)[rr].astype(float)
    beta = ru.platt_fit(s[tr], y[tr]); p = ru.platt_apply(beta, s)
    ABL_SCORES[name] = s                       # raw score for ranking (AUROC is Platt-invariant)
    e = ru.evaluate(y[te], p[te], subj[te], n_boot=N_BOOT, seed=SEED, label=name, rank_score=s[te])
    # mean belief in each stage by true KDIGO stage (latent-state validity, R1.8)
    occ = pd.DataFrame(B[C.is_test.to_numpy()], columns=STATE_LABELS).groupby(
        C.loc[C.is_test, "kdigo_stage"].to_numpy()).mean()
    occ.to_csv(REV / f"R3_mean_belief_by_true_stage__{name.replace(' ', '_').replace('(', '').replace(')', '')}.csv")
    rows.append(e)
    print(f"  {name:<32} AUROC {e.loc[e.metric=='AUROC','estimate'].item():.3f}  ({time.time()-t0:.0f} s)")
R3 = pd.concat(rows, ignore_index=True)
_d = []
for name in VARIANTS:
    if name == "Full model": continue
    d, lo, hi = ru.paired_delta_auc(y[te], ABL_SCORES[name][te], ABL_SCORES["Full model"][te],
                                    subj[te], n_boot=N_BOOT, seed=SEED)
    _d.append(dict(model=name, delta_auroc_vs_full=d, lo=lo, hi=hi))
R3d = pd.DataFrame(_d)
R3.to_csv(REV / "R3_ablation.csv", index=False); R3d.to_csv(REV / "R3_ablation_delta.csv", index=False)
print(ru.wide(R3)[["AUROC", "AUPRC", "Brier", "Slope"]].to_string())
print(R3d.round(3).to_string(index=False))
print("Momentum: not part of the AKI engine (respiratory only); ablated in RF_revision_IBMED.ipynb.")

  Full model                       AUROC 0.772  (78 s)
  Equal feature weights            AUROC 0.774  (86 s)
  Unordered transitions            AUROC 0.771  (84 s)
  No age-specific recovery         AUROC 0.772  (84 s)
  No temporal dynamics (static)    AUROC 0.771  (87 s)
  Population z-score emissions     AUROC 0.753  (89 s)
  No RRT absorbing state           AUROC 0.772  (86 s)
                                                AUROC                   AUPRC                   Brier                   Slope
model                                                                                                                        
Full model                     0.772 (0.768 to 0.778)  0.221 (0.211 to 0.230)  0.060 (0.057 to 0.062)  1.039 (0.987 to 1.093)
Equal feature weights          0.774 (0.769 to 0.780)  0.211 (0.202 to 0.220)  0.060 (0.057 to 0.062)  1.024 (0.967 to 1.079)
Unordered transitions          0.771 (0.767 to 0.777)  0.216 (0.207 to 0.226)  0.060 (0.057 to 0.062)  1.038 (0

In [9]:
# ---------------- R4 · baselines on the identical split, labels and horizons
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
try:
    import xgboost as xgb; HAVE_XGB = True
except Exception:
    HAVE_XGB = False
try:
    import torch; HAVE_TORCH = True
except Exception:
    HAVE_TORCH = False
print(f"xgboost: {HAVE_XGB} | torch (GRU): {HAVE_TORCH}")

MAX_TRAIN_ROWS = int(os.environ.get("REV_MAX_TRAIN", "600000"))   # rows sampled from training subjects
BASE_HORIZONS = tuple(int(h) for h in os.environ.get("REV_BASE_H", "12,24,48").split(","))

# causal history features (trailing windows within the stay; never look ahead)
t0 = time.time()
HF = pd.DataFrame(index=C.index)
g = C.groupby("stay_id", sort=False)
_roll = lambda ser, w: ser.groupby(C["stay_id"], sort=False).rolling(w, min_periods=1)
_flat = lambda r: r.reset_index(level=0, drop=True)
for f in FEATS:
    HF[f] = C[f]
    for w in (6, 24):
        HF[f"{f}_mean{w}"] = _flat(_roll(C[f], w).mean())
    HF[f"{f}_min24"] = _flat(_roll(C[f], 24).min())
    HF[f"{f}_max24"] = _flat(_roll(C[f], 24).max())
    HF[f"{f}_d6"] = C[f] - g[f].shift(6)
    HF[f"{f}_obs24"] = _flat(_roll(C[f].notna().astype(float), 24).sum())
HF["hours_in_icu"] = C["hour"]; HF["age"] = C["anchor_age"]
HF = HF.astype(np.float32)                                   # halves memory (about 0.8 GB on the full cohort)
INST = FEATS
HIST = list(HF.columns)
print(f"history features: {len(HIST)} built in {time.time()-t0:.0f} s")

# training subjects -> 80% fit / 20% recalibration (grouped), seed 7
_trs = np.array(sorted(set(C.subject_id) - TEST_SUBJ)); np.random.default_rng(SEED).shuffle(_trs)
CAL_SUBJ = set(_trs[: int(0.2 * len(_trs))])


def make_models():
    m = {
        "Logistic (current features)": (INST, make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                        LogisticRegression(max_iter=2000))),
        "Logistic (history features)": (HIST, make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                        StandardScaler(), LogisticRegression(max_iter=2000))),
        "Random forest (history)": (HIST, make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                    RandomForestClassifier(n_estimators=200, min_samples_leaf=50, max_features="sqrt",
                                                           n_jobs=-1, random_state=SEED))),
    }
    if HAVE_XGB:
        m["XGBoost (history)"] = (HIST, xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                                  n_jobs=-1, random_state=SEED, eval_metric="logloss"))
    else:
        m["Gradient boosting (history)"] = (HIST, HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                            max_leaf_nodes=63, early_stopping=False, random_state=SEED))
    return m


def gru_predict(H, fit_rows, cal_rows, test_rows, lookback=24, epochs=4, hidden=32):
    """GRU over the last `lookback` hours (values + observation masks), causal."""
    torch.manual_seed(SEED)
    Xv = C[FEATS].to_numpy(np.float32)
    _trm = ~C["is_test"].to_numpy()
    mu = np.nanmean(Xv[_trm], 0); sd = np.nanstd(Xv[_trm], 0) + 1e-6
    Z = np.concatenate([np.nan_to_num((Xv - mu) / sd), (~np.isnan(Xv)).astype(np.float32)], 1)

    def windows(rows):
        idx = rows[:, None] + np.arange(-lookback + 1, 1)[None]
        valid = idx >= ROW_START[rows][:, None]
        W = Z[np.where(valid, idx, rows[:, None])] * valid[..., None]
        return np.concatenate([W, valid[..., None].astype(np.float32)], 2)

    class Net(torch.nn.Module):
        def __init__(s):
            super().__init__(); s.g = torch.nn.GRU(Z.shape[1] + 1, hidden, batch_first=True)
            s.o = torch.nn.Linear(hidden, 1)
        def forward(s, x):
            return s.o(s.g(x)[1][-1]).squeeze(-1)
    net = Net(); opt = torch.optim.Adam(net.parameters(), 1e-3); lossf = torch.nn.BCEWithLogitsLoss()
    yall = np.zeros(len(C), np.float32); yall[LAB[H]._row.to_numpy()] = LAB[H].y_escalate.to_numpy()
    rng = np.random.default_rng(SEED)
    for ep in range(epochs):
        perm = rng.permutation(fit_rows)
        for a in range(0, len(perm), 2048):
            r = perm[a:a + 2048]
            xb = torch.from_numpy(windows(r)); yb = torch.from_numpy(yall[r])
            opt.zero_grad(); l = lossf(net(xb), yb); l.backward(); opt.step()
    net.eval(); out = {}
    with torch.no_grad():
        for nm, rows in [("cal", cal_rows), ("test", test_rows)]:
            out[nm] = np.concatenate([torch.sigmoid(net(torch.from_numpy(windows(rows[a:a + 20000])))).numpy()
                                      for a in range(0, len(rows), 20000)])
    return out["cal"], out["test"]


rows, deltas, TIMES, PRED24 = [], [], [], {}
for H in BASE_HORIZONS:
    L = LAB[H]
    trn = L[~L.is_test]; te = L[L.is_test]
    fit_df = trn[~trn.subject_id.isin(CAL_SUBJ)]; cal_df = trn[trn.subject_id.isin(CAL_SUBJ)]
    if len(fit_df) > MAX_TRAIN_ROWS:
        fit_df = fit_df.sample(MAX_TRAIN_ROWS, random_state=SEED)
    yte, ste = te.y_escalate.to_numpy(), te.subject_id.to_numpy()
    preds = {"Belief state (Platt)": te.p_cal.to_numpy()}
    RAW_BELIEF = te.risk.to_numpy()
    for name, (cols, model) in make_models().items():
        t0 = time.time()
        model.fit(HF.loc[fit_df._row, cols].to_numpy(np.float32), fit_df.y_escalate.to_numpy())
        t_fit = time.time() - t0
        pc = model.predict_proba(HF.loc[cal_df._row, cols].to_numpy(np.float32))[:, 1]
        t0 = time.time()
        pt = model.predict_proba(HF.loc[te._row, cols].to_numpy(np.float32))[:, 1]
        t_pred = time.time() - t0
        lg = lambda v: np.log(np.clip(v, 1e-6, 1 - 1e-6) / (1 - np.clip(v, 1e-6, 1 - 1e-6)))
        beta = ru.platt_fit(lg(pc), cal_df.y_escalate.to_numpy())
        preds[name] = ru.platt_apply(beta, lg(pt))
        TIMES.append(dict(horizon_h=H, model=name, fit_s=t_fit, predict_s=t_pred, n_fit=len(fit_df),
                          n_test=len(te), predict_us_per_row=1e6 * t_pred / len(te)))
        print(f"  H={H} {name:<30} fit {t_fit:5.0f} s")
    if HAVE_TORCH:
        t0 = time.time()
        try:
            pc, pt = gru_predict(H, fit_df._row.to_numpy(), cal_df._row.to_numpy(), te._row.to_numpy())
            lg = lambda v: np.log(np.clip(v, 1e-6, 1 - 1e-6) / (1 - np.clip(v, 1e-6, 1 - 1e-6)))
            preds["GRU (24 h lookback)"] = ru.platt_apply(ru.platt_fit(lg(pc), cal_df.y_escalate.to_numpy()), lg(pt))
            TIMES.append(dict(horizon_h=H, model="GRU (24 h lookback)", fit_s=time.time() - t0))
        except Exception as ex:                       # optional model: never block the other results
            print(f"  GRU skipped at H={H}: {type(ex).__name__}: {ex}")
    for name, p in preds.items():
        rk = RAW_BELIEF if name == "Belief state (Platt)" else None
        rows.append(ru.evaluate(yte, p, ste, n_boot=N_BOOT, seed=SEED, label=f"{H} h | {name}", rank_score=rk))
        if name != "Belief state (Platt)":
            d, lo, hi = ru.paired_delta_auc(yte, p, RAW_BELIEF, ste, n_boot=N_BOOT, seed=SEED)
            deltas.append(dict(horizon_h=H, model=name, delta_auroc_vs_belief=d, lo=lo, hi=hi))
    if H == 24:
        PRED24 = dict(preds, y=yte)
R4 = pd.concat(rows, ignore_index=True); R4d = pd.DataFrame(deltas); R4t = pd.DataFrame(TIMES)
R4.to_csv(REV / "R4_baselines.csv", index=False); R4d.to_csv(REV / "R4_baselines_delta.csv", index=False)
R4t.to_csv(REV / "R4_baselines_runtime.csv", index=False)
print(ru.wide(R4)[["AUROC", "AUPRC", "Brier", "Slope"]].to_string())
print(R4d.round(3).to_string(index=False))

# decision curve at 24 h (test hours)
if PRED24:
    ths = np.round(np.arange(0.01, 0.41, 0.01), 2)
    plt.figure(figsize=(6, 4.5)); dca = []
    for name, p in PRED24.items():
        if name == "y": continue
        nb = ru.net_benefit(PRED24["y"], p, ths); nb["model"] = name; dca.append(nb)
        plt.plot(ths, nb.net_benefit, label=name)
    plt.plot(ths, nb.treat_all, "k:", label="alert on every hour"); plt.axhline(0, color="grey", lw=0.8)
    plt.ylim(-0.01, max(0.02, PRED24["y"].mean() * 1.1)); plt.xlabel("Threshold probability")
    plt.ylabel("Net benefit"); plt.legend(fontsize=7, frameon=False); plt.tight_layout()
    plt.savefig(FIGR / "aki_decision_curve_24h.png", dpi=200); plt.close()
    pd.concat(dca).to_csv(REV / "R2b_decision_curve_24h.csv", index=False)

xgboost: True | torch (GRU): True
history features: 30 built in 23 s
  H=12 Logistic (current features)    fit     0 s
  H=12 Logistic (history features)    fit     2 s
  H=12 Random forest (history)        fit    23 s
  H=12 XGBoost (history)              fit     3 s
  H=24 Logistic (current features)    fit     0 s
  H=24 Logistic (history features)    fit     2 s
  H=24 Random forest (history)        fit    22 s
  H=24 XGBoost (history)              fit     3 s
  H=48 Logistic (current features)    fit     0 s
  H=48 Logistic (history features)    fit     2 s
  H=48 Random forest (history)        fit    22 s
  H=48 XGBoost (history)              fit     3 s
                                                     AUROC                   AUPRC                   Brier                   Slope
model                                                                                                                             
12 h | Belief state (Platt)         0.810 (0.806 to 0.816)  0.167 (0.

In [10]:
# ------------------------------- R5 · temporal validation (MIMIC-IV anchor_year_group)
def _find_patients_csv(*dirs):
    """MIMIC-IV hosp/patients as .csv or .csv.gz; MIMIC_HOSP env var overrides the search."""
    cands = [os.environ.get("MIMIC_HOSP", "")] + [str(d) for d in dirs] + [
        "/Users/sumiyasayeed/Documents/mimic4/mimiciv/hosp", "/Users/sumiyasayeed/Documents/mimic4/hosp"]
    for d in cands:
        for ext in (".csv", ".csv.gz"):
            if d and (Path(d) / f"patients{ext}").exists():
                return Path(d) / f"patients{ext}"
    hits = sorted(Path("/Users/sumiyasayeed/Documents/mimic4").glob("**/patients.csv*"))
    return hits[0] if hits else None


H = 24
if SYNTHETIC_TEST:
    _subs = C.subject_id.unique()
    AYG = pd.Series(np.random.default_rng(1).choice(["2008 - 2010", "2011 - 2013", "2014 - 2016",
                                                     "2017 - 2019"], len(_subs)), index=_subs)
else:
    _pc = _find_patients_csv(HOSP_DIR)
    if _pc is None:
        raise FileNotFoundError("patients.csv(.gz) not found. Run  os.environ['MIMIC_HOSP'] = "
                                "'/path/to/mimiciv/hosp'  and rerun this cell.")
    print("patients table:", _pc)
    AYG = ru.anchor_year_groups(_pc)
AYG = AYG.reindex(C.subject_id.unique())
DEV_G, TEST_G = ru.temporal_split(AYG, 0.10)
print("development groups:", DEV_G, "| temporal test groups:", TEST_G)
print(AYG.value_counts().sort_index().to_string())
L = LAB[H].copy(); L["ayg"] = L.subject_id.map(AYG)
dev, tt = L[L.ayg.isin(DEV_G)], L[L.ayg.isin(TEST_G)]
beta_dev = ru.platt_fit(dev.risk.to_numpy(), dev.y_escalate.to_numpy())
rows = [ru.evaluate(tt.y_escalate, ru.platt_apply(beta_dev, tt.risk.to_numpy()), tt.subject_id,
                    n_boot=N_BOOT, seed=SEED, label="Belief state (Platt fit on development years)",
                    rank_score=tt.risk.to_numpy())]
# best history baseline, refit on development years only
fit_df = dev[~dev.subject_id.isin(CAL_SUBJ)]; cal_df = dev[dev.subject_id.isin(CAL_SUBJ)]
if len(fit_df) > MAX_TRAIN_ROWS: fit_df = fit_df.sample(MAX_TRAIN_ROWS, random_state=SEED)
for name, (cols, model) in make_models().items():
    if name.startswith("Random forest"): continue          # keep R5 fast; RF reported in R4
    model.fit(HF.loc[fit_df._row, cols].to_numpy(np.float32), fit_df.y_escalate.to_numpy())
    lg = lambda v: np.log(np.clip(v, 1e-6, 1 - 1e-6) / (1 - np.clip(v, 1e-6, 1 - 1e-6)))
    beta = ru.platt_fit(lg(model.predict_proba(HF.loc[cal_df._row, cols].to_numpy(np.float32))[:, 1]),
                        cal_df.y_escalate.to_numpy())
    p = ru.platt_apply(beta, lg(model.predict_proba(HF.loc[tt._row, cols].to_numpy(np.float32))[:, 1]))
    rows.append(ru.evaluate(tt.y_escalate, p, tt.subject_id, n_boot=N_BOOT, seed=SEED, label=name))
R5 = pd.concat(rows, ignore_index=True)
R5["dev_groups"] = "; ".join(DEV_G); R5["test_groups"] = "; ".join(TEST_G)
R5.to_csv(REV / "R5_temporal_validation.csv", index=False)
print(f"temporal test: {tt.subject_id.nunique():,} subjects, {len(tt):,} hours, prevalence {tt.y_escalate.mean():.3f}")
print(ru.wide(R5).to_string())

patients table: /Users/sumiyasayeed/Documents/mimic4/mimiciv/hosp/patients.csv.gz
development groups: ['2008 - 2010', '2011 - 2013', '2014 - 2016', '2017 - 2019'] | temporal test groups: ['2020 - 2022']
anchor_year_group
2008 - 2010    10754
2011 - 2013     8145
2014 - 2016     8232
2017 - 2019     7810
2020 - 2022     5817
temporal test: 5,815 subjects, 838,965 hours, prevalence 0.053
                                                                AUROC                   AUPRC                   Brier                       CITL                   Slope                     E/O
model                                                                                                                                                                                           
Belief state (Platt fit on development years)  0.750 (0.739 to 0.760)  0.180 (0.165 to 0.198)  0.050 (0.047 to 0.052)  -0.260 (-0.322 to -0.197)  0.989 (0.910 to 1.085)  1.271 (1.200 to 1.347)
Logistic (current features)     

In [11]:
# ---------------------------------------- R6 · attribution fidelity and stability
# Exact identity of the filter: for any two states i, j at hour t,
#   log[a_t(j)/a_t(i)] = log[pred_t(j)/pred_t(i)] + sum_f LLR_f(j vs i)
# so the per-feature LLR is an exact additive decomposition of the evidence update.
# Below: (a) numerical check of the identity, (b) deletion test on the reported risk scale,
# (c) rank agreement LLR vs deletion effect, (d) stability under small measurement noise.
H = 24
X, M = obs_matrix("clinical")
L = LAB[H][LAB[H].is_test]
n_s = min(50_000, len(L))
samp = L.sample(n_s, random_state=SEED)["_row"].to_numpy()
FL = feature_loglik(X[samp], M[samp], CLINICAL_MEANS, CLINICAL_SCALES, W_FULL)   # (n, K, F)
bank = transition_bank("tridiag")
prev_b = np.where((samp - ROW_START[samp])[:, None] > 0, B_FULL_RAW[np.maximum(samp - 1, 0)], np.nan)
is_first = (samp - ROW_START[samp]) == 0
Tst = bank[STAY_AGE_IDX[np.searchsorted(STARTS, samp, side="right") - 1]]
pred = np.where(is_first[:, None], PRIOR_BELIEF[None], np.einsum("nk,nkj->nj", np.nan_to_num(prev_b), Tst))
post = B_FULL_RAW[samp]


def posterior(pred, logE):
    lp = np.log(pred + 1e-30) + logE; lp -= lp.max(1, keepdims=True)
    p = np.exp(lp); return p / p.sum(1, keepdims=True)


# for t = 0 the engine applies no prediction step: pred = prior (handled above)
recon = posterior(pred, FL.sum(2))
ident_err = float(np.max(np.abs(recon - post)))
cur = post.argmax(1); to = np.minimum(cur + 1, N_STATES - 1)
LLR = FL[np.arange(n_s), to, :] - FL[np.arange(n_s), cur, :]          # (n, F), paper's attribution
risk = post[:, 2] + post[:, 3]
DEL = np.zeros_like(LLR)
for j in range(len(FEATS)):
    keep = np.ones(len(FEATS), bool); keep[j] = False
    DEL[:, j] = risk - (lambda q: q[:, 2] + q[:, 3])(posterior(pred, FL[:, :, keep].sum(2)))
obs = M[samp] > 0
nobs = obs.sum(1)
use = nobs >= 2
top_llr = np.where(obs, LLR, -np.inf).argmax(1)
top_del = np.where(obs, DEL, -np.inf).argmax(1)
top_abs_del = np.where(obs, np.abs(DEL), -np.inf).argmax(1)
from scipy.stats import spearmanr
rho = [spearmanr(LLR[i, obs[i]], DEL[i, obs[i]]).correlation for i in np.nonzero(nobs >= 3)[0][:20000]]
moved = np.abs(DEL).max(1) > 1e-3
# stability: 5% of the stage-0 anchor SD as Gaussian measurement noise
rng = np.random.default_rng(SEED)
Xn = X[samp] + rng.normal(0, 0.05 * CLINICAL_SCALES[0][None, :], size=X[samp].shape) * M[samp]
FLn = feature_loglik(Xn, M[samp], CLINICAL_MEANS, CLINICAL_SCALES, W_FULL)
LLRn = FLn[np.arange(n_s), to, :] - FLn[np.arange(n_s), cur, :]
top_n = np.where(obs, LLRn, -np.inf).argmax(1)
R6 = pd.DataFrame([
    dict(check="max |reconstructed - engine posterior| (exact identity)", value=ident_err),
    dict(check="hours sampled (test, below stage 2)", value=n_s),
    dict(check="hours with >=2 observed features", value=int(use.sum())),
    dict(check="top-LLR feature = feature whose deletion lowers risk most", value=float((top_llr == top_del)[use].mean())),
    dict(check="same, restricted to hours where deletion moves risk by >0.001", value=float((top_llr == top_del)[use & moved].mean())),
    dict(check="top-LLR feature = feature with largest |deletion effect|", value=float((top_llr == top_abs_del)[use].mean())),
    dict(check="median Spearman(LLR, deletion effect) across features, hours with >=3 features", value=float(np.nanmedian(rho))),
    dict(check="sign agreement LLR vs deletion effect (feature-hours, |effect|>0.001)",
         value=float((np.sign(LLR) == np.sign(DEL))[obs & (np.abs(DEL) > 1e-3)].mean())),
    dict(check="top-driver unchanged under 5% measurement noise", value=float((top_n == top_llr)[use].mean())),
])
R6.to_csv(REV / "R6_attribution_fidelity.csv", index=False)
print(R6.to_string(index=False))

                                                                         check        value
                       max |reconstructed - engine posterior| (exact identity)     0.000005
                                           hours sampled (test, below stage 2) 50000.000000
                                              hours with >=2 observed features 49827.000000
                     top-LLR feature = feature whose deletion lowers risk most     0.802236
                 same, restricted to hours where deletion moves risk by >0.001     0.933293
                      top-LLR feature = feature with largest |deletion effect|     0.014069
median Spearman(LLR, deletion effect) across features, hours with >=3 features     1.000000
         sign agreement LLR vs deletion effect (feature-hours, |effect|>0.001)     0.955848
                               top-driver unchanged under 5% measurement noise     0.983503


In [12]:
# ----------------------------------------- R7 · cost and reproducibility record
n_hand = CLINICAL_MEANS.size + CLINICAL_SCALES.size + len(FEATS) + 1 + 4 + (N_STATES - 1)
R7 = dict(
    engine_seconds_full_cohort_incl_compile=T_ENGINE, engine_us_per_stay_hour=1e6 * T_ENGINE / len(C),
    jax_engine_seconds_per_stay_compiled=T_JAX_PER_STAY, batch_vs_published_max_abs_diff=MAXDIFF,
    engine_parameters_hand_specified=int(n_hand), engine_parameters_fitted=2,
    stays=int(C.stay_id.nunique()), stay_hours=int(len(C)), test_subjects=len(TEST_SUBJ),
    split_seed=SEED, bootstrap_replicates=N_BOOT, environment=ru.env_versions(),
)
json.dump(R7, open(REV / "R7_runtime_and_environment.json", "w"), indent=2, default=str)
print(json.dumps(R7, indent=2, default=str))
print("\nAll revision outputs written to", REV.resolve())

{
  "engine_seconds_full_cohort_incl_compile": 5.182121276855469,
  "engine_us_per_stay_hour": 0.7832555426812844,
  "jax_engine_seconds_per_stay_compiled": 0.0006125855445861816,
  "batch_vs_published_max_abs_diff": 0.0,
  "engine_parameters_hand_specified": 44,
  "engine_parameters_fitted": 2,
  "stays": 52742,
  "stay_hours": 6616131,
  "test_subjects": 12227,
  "split_seed": 7,
  "bootstrap_replicates": 200,
  "environment": {
    "python": "3.13.5",
    "platform": "macOS-27.0.1-arm64-arm-64bit-Mach-O",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "scipy": "1.18.1",
    "sklearn": "1.9.1",
    "jax": "0.10.0",
    "xgboost": "3.2.0",
    "torch": "2.13.0",
    "lifelines": "0.30.3"
  }
}

All revision outputs written to /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/PEMDT-AKI/results/revision_v2
